# 00 - Preparación de ambiente

Objetivo: preparar únicamente el entorno lógico del proyecto **fintech-medallion-databricks**.

Este notebook crea el catálogo y los esquemas requeridos por la arquitectura medallion:

- `bronze`
- `silver`
- `gold`

In [ ]:
# Widgets de configuración usados por Databricks Asset Bundles
# Valores alineados con databricks.yml

dbutils.widgets.text("catalog_name", "fintech_lakehouse", "Unity Catalog")
dbutils.widgets.text("bronze_schema", "bronze", "Schema Bronze")
dbutils.widgets.text("silver_schema", "silver", "Schema Silver")
dbutils.widgets.text("gold_schema", "gold", "Schema Gold")
dbutils.widgets.text("raw_base_path", "abfss://raw@<storage-account>.dfs.core.windows.net/fintech", "ADLS Raw Base Path")

catalog_name = dbutils.widgets.get("catalog_name")
bronze_schema = dbutils.widgets.get("bronze_schema")
silver_schema = dbutils.widgets.get("silver_schema")
gold_schema = dbutils.widgets.get("gold_schema")
raw_base_path = dbutils.widgets.get("raw_base_path")

print(f"Catalog: {catalog_name}")
print(f"Bronze schema: {bronze_schema}")
print(f"Silver schema: {silver_schema}")
print(f"Gold schema: {gold_schema}")
print(f"Raw base path: {raw_base_path}")

In [ ]:
# Validaciones básicas de parámetros

required_values = {
    "catalog_name": catalog_name,
    "bronze_schema": bronze_schema,
    "silver_schema": silver_schema,
    "gold_schema": gold_schema,
}

for key, value in required_values.items():
    if not value or not value.strip():
        raise ValueError(f"El parámetro {key} no puede estar vacío")

# raw_base_path puede mantener placeholder en ambientes académicos,
# pero se recomienda reemplazarlo por el storage real antes de ejecutar ingesta productiva.
if "<storage-account>" in raw_base_path:
    print("ADVERTENCIA: raw_base_path usa placeholder <storage-account>. Actualizar en ambiente real.")

In [ ]:
# Crear catálogo y esquemas de la arquitectura medallion

spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog_name}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{bronze_schema}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{silver_schema}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{gold_schema}")

spark.sql(f"USE CATALOG {catalog_name}")

print("Ambiente medallion preparado correctamente")
print(f"- {catalog_name}.{bronze_schema}")
print(f"- {catalog_name}.{silver_schema}")
print(f"- {catalog_name}.{gold_schema}")

In [ ]:
# Evidencia rápida de objetos creados

schemas_df = spark.sql(f"SHOW SCHEMAS IN {catalog_name}")
display(schemas_df)

## Resultado esperado

Al finalizar este notebook deben existir el catálogo y los tres esquemas principales del lakehouse fintech.

La seguridad y los permisos no se configuran aquí para mantener separación de responsabilidades.